# Fundamentals 02 - Skill API

Objetivo: empaquetar Tools, prompts, contrato y policy como una capacidad reusable mediante `toolkit.skill`.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_DEMO_SYMBOL | skill | Entrada del usuario para las Tools de la Skill. |
| tools | API inspectors | Capacidades reales agrupadas por toolkit.skill. |
| runtime | python-runtime | Ejecucion reproducible antes de cambiar provider. |

In [ ]:
import os

import agentic_systems as toolkit

SYMBOL = os.getenv("AGENTIC_SYSTEMS_DEMO_SYMBOL", "skill")

## 1) Tools de la capacidad

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.PUBLIC_API}

@toolkit.tool
def package_version() -> dict:
    return {"package_version": toolkit.__version__}

## 2) Crear y validar la Skill

La Skill conserva componentes declarativos; no ejecuta un provider al construirse.

In [ ]:
inspection_skill = toolkit.skill(
    name="public_api_inspection",
    version="1.0.0",
    description="Inspecciona la superficie publica instalada.",
    tools=[inspect_public_api, package_version],
    prompts={"instructions": "Usa Tools para responder con evidencia instalada."},
    contracts={"default": toolkit.AgentContract(must_call=["inspect_public_api"]).model_dump(mode="json")},
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2).model_dump(mode="json"),
    metadata={"domain": "public-api"},
)
validation = inspection_skill.check()
toolkit.show_json({
    "info": inspection_skill.info(),
    "validation": validation.to_dict(),
    "tool_names": inspection_skill.tool_names,
}, title="Skill contract")

## 3) Consumir la Skill desde un Agent

In [ ]:
runtime = toolkit.runtime(provider="python-runtime")
agent = toolkit.agent(
    name="skill_consumer",
    instructions=inspection_skill.instructions,
    skills=[inspection_skill],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
)
result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": SYMBOL}},
    mode="eval",
)
toolkit.human_result(result, title="Skill Agent RunResult", show_lineage=True)

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "toolkit.tool", "toolkit.skill", "Skill.check", "Skill.info", "Skill.tool_names",
    "toolkit.runtime", "toolkit.agent", "agent.run", "toolkit.human_result",
]
toolkit.show_json(api_coverage, title="Skill API coverage")

## Resultado esperado

Una Skill valida y un Agent que ejecuta una Tool registrada por la Skill.